In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Flow tube-state: fixed five conditions

Local static handoff, not an executed result. Run all runs OFF, TERMINAL_A/B, FLOW_A/B at the fixed prompt and seed. This notebook contains its complete unpublished implementation snapshot, rather than fetching an old main commit. GitHub publication and binding to a new immutable release SHA remain a separate authorized action.

50 steps; guidance only at indices 44/45/46, then normal 47/48/49. No model or VAE backward. R is fixed by same-run terminal A/B before Flow. Fixed budget: 124 Transformer forwards, 62 real and 12 shadow scheduler steps, 5 VAE decodes, 5 normal MP4 saves, 20 receiver encodes. Failures stay in all five rows. First-round saved RGB and residual temporal MSE limits are 1.5 times the corresponding terminal reference; these are not perceptual thresholds.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import base64, io, json, os, signal, subprocess, sys, zipfile
RUN_ID = 'flow_tube_state_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SOURCE = Path('/content') / (RUN_ID + '_source')
SOURCE.mkdir(exist_ok=False)
PAYLOAD = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:
    archive.extractall(SOURCE)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
subprocess.run(['ffmpeg', '-version'], check=True)
subprocess.run(['ffprobe', '-version'], check=True)
print('Local bundled source:', SOURCE)


In [ ]:
OUTPUT = Path('/content/drive/MyDrive/Video-WM/FlowTubeState') / RUN_ID
CONFIG = SOURCE / 'experiments/wan_state_clock/configs/flow_replication.json'
LOG = OUTPUT.parent / f'{RUN_ID}.launcher.log'
LOG.parent.mkdir(parents=True, exist_ok=True)
# Retain exact source beside the results without a published-source claim.
SOURCE_ARCHIVE = OUTPUT.parent / f'{RUN_ID}.source.zip'
SOURCE_ARCHIVE.write_bytes(base64.b64decode(PAYLOAD))
cmd = [sys.executable, '-u', '-m', 'experiments.wan_state_clock.flow_run', '--config', str(CONFIG), '--output', str(OUTPUT)]
with LOG.open('w') as log:
    process = subprocess.Popen(cmd, cwd=SOURCE, start_new_session=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        code = process.wait()
    except BaseException:
        try:
            process.send_signal(signal.SIGTERM)
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
        except ProcessLookupError:
            pass
        raise
print('Exit:', code, 'Results:', OUTPUT, 'Log:', LOG)
if (OUTPUT / 'result.json').exists():
    result = json.loads((OUTPUT / 'result.json').read_text())
    print(json.dumps({k: result.get(k) for k in ('status', 'actual_calls', 'first_round_criteria_met', 'saved_quality_comparisons', 'resources', 'failures')}, indent=2))
if code:
    raise subprocess.CalledProcessError(code, cmd)
